In [ ]:
pip install transformers torch scikit-learn

In [ ]:
# === 1. IMPORT CÁC THƯ VIỆN CẦN THIẾT ===
import json
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
from collections import Counter
import torch.nn as nn
from transformers import BertModel, BertPreTrainedModel
import os
import numpy as np
import random
import time

In [ ]:
# 1. Đường dẫn tới file train.json của bạn
train_path = '/kaggle/input/datasets/daishinkan002/new-york-times-relation-extraction-dataset/dataset/train.json'

# Khởi tạo bộ đếm
relation_counts = Counter()
entity_counts = Counter()

# 2. Đọc file và tiến hành đếm
with open(train_path, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        
        # Đếm Quan hệ (Relations)
        rels = data.get('relationMentions', [])
        if rels:
            relation_counts[rels[0]['label']] += 1
        else:
            relation_counts['NA'] += 1 # Nhãn NA cho câu không có quan hệ
            
        # Đếm Thực thể (Entities)
        ents = data.get('entityMentions', [])
        for ent in ents:
            entity_counts[ent['label']] += 1

# --- 3. VẼ BIỂU ĐỒ TRỰC QUAN HÓA ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Biểu đồ 1: Phân phối Thực thể (Entity)
ent_labels, ent_values = zip(*entity_counts.most_common())
ax1.bar(ent_labels, ent_values, color=['#3498db', '#e74c3c', '#2ecc71', '#9b59b6'])
ax1.set_title('Phân phối số lượng Thực thể (Entities)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Số lượng mẫu', fontsize=10)
# Hiển thị số liệu trên đầu cột
for i, v in enumerate(ent_values):
    ax1.text(i, v + 500, str(v), ha='center', fontweight='bold')

# Biểu đồ 2: Top 10 Quan hệ phổ biến nhất (Relation)
# Bỏ qua nhãn NA để xem rõ các quan hệ thực sự, lấy top 10
rel_common = [item for item in relation_counts.most_common(11) if item[0] != 'NA'][:10]
rel_labels, rel_values = zip(*rel_common)

# Vẽ biểu đồ thanh ngang (barh) vì tên quan hệ rất dài
ax2.barh(rel_labels, rel_values, color='#e67e22')
ax2.set_title('Top 10 Quan hệ phổ biến nhất (bỏ qua NA)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Số lượng mẫu', fontsize=10)
ax2.invert_yaxis() # Đảo ngược trục Y để Top 1 nằm trên cùng

plt.tight_layout()
plt.show()

In [ ]:
# Danh sách các quan hệ phổ biến trong bộ dữ liệu NYT
# 'NA' thường được dùng cho các cặp thực thể không có quan hệ (nhãn mặc định)
rel2id = {
    "NA": 0,
    "/business/company/founders": 1,
    "/business/company/advisors": 2,
    "/business/person/company": 4,
    "/location/administrative_division/country": 5,
    "/location/country/capital": 6,
    "/location/country/administrative_divisions": 7,
    "/location/location/contains": 8,
    "/people/deceased_person/place_of_death": 10,
    "/people/person/nationality": 11,
    "/people/person/place_of_birth": 12,
    "/people/person/place_lived": 13,
    "/people/person/religion": 14,
    "/sports/sports_team/location": 17,
    "/sports/sports_team_roster/team": 18
}

# Tạo ngược lại id2rel để sau này giải mã kết quả dự đoán (tùy chọn)
id2rel = {v: k for k, v in rel2id.items()}

In [ ]:
# Danh mục nhãn Thực thể theo chuẩn BIO
ent2id = {
    "O": 0,         # Outside: Không phải thực thể
    "B-PER": 1,     # Begin - Person
    "I-PER": 2,     # Inside - Person
    "B-ORG": 3,     # Begin - Organization
    "I-ORG": 4,     # Inside - Organization
    "B-LOC": 5,     # Begin - Location
    "I-LOC": 6      # Inside - Location
}

# Ánh xạ từ khóa trong file JSON sang tiền tố chuẩn
type_map = {
    "PERSON": "PER",
    "ORGANIZATION": "ORG",
    "LOCATION": "LOC"
}

In [ ]:


# === 2. ĐỊNH NGHĨA LỚP DỮ LIỆU ĐÃ NÂNG CẤP (JOINT EXTRACTION) ===
class NYTDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_len, rel2id, ent2id):
        self.data = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                self.data.append(json.loads(line))
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rel2id = rel2id
        self.ent2id = ent2id

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['sentText']
        
        # 1. Tokenize câu văn
        encoding = self.tokenizer(
            text, max_length=self.max_len, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        input_ids = encoding['input_ids'].flatten()
        
        # 2. Xử lý nhãn Quan hệ (Relation)
        rels = item.get('relationMentions', [])
        rel_label = rels[0]['label'] if rels else 'NA'
        rel_id = self.rel2id.get(rel_label, 0)

        # 3. XỬ LÝ NHÃN THỰC THỂ (NER) theo chuẩn BIO
        entity_labels = torch.zeros(self.max_len, dtype=torch.long) # Mặc định là nhãn 'O' (số 0)
        entities = item.get('entityMentions', [])
        
        for ent in entities:
            ent_text = ent.get('text', '')
            ent_type = ent.get('label', '') # VD: 'PERSON'
            
            mapped_type = type_map.get(ent_type, "ORG") # Mặc định là ORG nếu tên lạ
            b_tag = self.ent2id.get(f"B-{mapped_type}", 0)
            i_tag = self.ent2id.get(f"I-{mapped_type}", 0)
            
            # Tokenize riêng cái tên thực thể
            ent_tokens = self.tokenizer.encode(ent_text, add_special_tokens=False)
            ent_len = len(ent_tokens)
            
            # Quét qua câu để tìm vị trí của thực thể và gán nhãn
            for i in range(len(input_ids) - ent_len + 1):
                if input_ids[i:i+ent_len].tolist() == ent_tokens:
                    entity_labels[i] = b_tag # Từ đầu tiên là B
                    for j in range(1, ent_len):
                        entity_labels[i+j] = i_tag # Các từ sau là I
                    break 

        return {
            'input_ids': input_ids,
            'attention_mask': encoding['attention_mask'].flatten(),
            'relation_labels': torch.tensor(rel_id, dtype=torch.long),
            'entity_labels': entity_labels # Đã có nhãn cho Thực thể!
        }

In [ ]:

class JointBERTForRE(nn.Module):
    def __init__(self, model_name, num_entity_labels, num_relation_labels):
        super(JointBERTForRE, self).__init__()
        # 1. Khởi tạo Transformer Backbone (BERT)
        self.bert = BertModel.from_pretrained(model_name)
        self.hidden_size = self.bert.config.hidden_size
        
        # 2. Nhánh Entity (NER Head) - Phân loại từng token
        self.entity_classifier = nn.Linear(self.hidden_size, num_entity_labels)
        
        # 3. Nhánh Relation (RE Head) - Phân loại cặp thực thể
        # Ở phiên bản đơn giản, ta dùng vector [CLS] làm đại diện ngữ cảnh câu
        self.relation_classifier = nn.Linear(self.hidden_size, num_relation_labels)
        
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        # Truyền qua BERT
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        
        sequence_output = outputs[0]  # [batch_size, seq_len, hidden_size]
        pooled_output = outputs[1]    # [batch_size, hidden_size] - Đại diện cho cả câu (CLS token)

        # Nhánh 1: Dự đoán thực thể (dùng sequence_output)
        entity_logits = self.entity_classifier(self.dropout(sequence_output))
        
        # Nhánh 2: Dự đoán quan hệ (dùng pooled_output)
        # Trong thực tế, bạn có thể kết hợp vector của 2 thực thể cụ thể để chính xác hơn
        relation_logits = self.relation_classifier(self.dropout(pooled_output))
        
        return entity_logits, relation_logits

In [ ]:
criterion_entity = nn.CrossEntropyLoss()
criterion_relation = nn.CrossEntropyLoss()

def compute_loss(entity_logits, entity_labels, relation_logits, relation_labels):
    # Loss cho thực thể (NER)
    # Cần flatten logits và labels để tính CrossEntropy
    loss_ent = criterion_entity(entity_logits.view(-1, num_entity_labels), entity_labels.view(-1))
    
    # Loss cho quan hệ (RE)
    loss_rel = criterion_relation(relation_logits, relation_labels)
    
    # Tổng loss (có thể điều chỉnh trọng số nếu cần)
    return loss_ent + loss_rel

In [ ]:
# 1. Khai báo các thông số cấu hình
num_entity_labels = 7 # Ví dụ: B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, O
num_relation_labels = 25 # 24 quan hệ của NYT + 1 nhãn 'None'
model_name = 'bert-base-cased'

# 2. KHỞI TẠO MODEL (Đây là bước bạn đang thiếu)
model = JointBERTForRE(model_name, num_entity_labels, num_relation_labels)

# Đưa model lên GPU nếu có
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 3. Bây giờ mới khởi tạo Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# 4. Các biến hỗ trợ khác
num_epochs = 5
history = {"loss": [], "ent_acc": [], "rel_acc": []}

In [ ]:
from transformers import BertTokenizer

# Khởi tạo Tokenizer khớp với model 'bert-base-cased'
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

# Bây giờ mới chạy dòng này
train_dataset = NYTDataset('/kaggle/input/datasets/daishinkan002/new-york-times-relation-extraction-dataset/dataset/train.json', tokenizer, max_len=128, rel2id=rel2id, ent2id=ent2id)

In [ ]:
from torch.utils.data import DataLoader

# Đường dẫn chính xác từ ảnh của bạn
path_to_json = '/kaggle/input/datasets/daishinkan002/new-york-times-relation-extraction-dataset/dataset/train.json'

# Khởi tạo lại
train_dataset = NYTDataset(path_to_json, tokenizer, max_len=128, rel2id=rel2id,ent2id=ent2id)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

print(f"Sẵn sàng! Đã nạp {len(train_dataset)} mẫu với các cột: {list(train_dataset.data[0].keys())}")

In [ ]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        # Chuyển TOÀN BỘ lên thiết bị
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        rel_labels = batch['relation_labels'].to(device)
        ent_labels = batch['entity_labels'].to(device) # Đưa nhãn thực thể lên GPU

        optimizer.zero_grad()
        
        # Forward pass: 1 lần chạy, ra 2 kết quả
        ent_logits, rel_logits = model(input_ids, attention_mask)
        
        # TÍNH LOSS TỔNG HỢP (Multi-task Learning)
        loss = compute_loss(ent_logits, ent_labels, rel_logits, rel_labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    history["loss"].append(avg_loss)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss tổng hợp: {avg_loss:.4f}")

In [ ]:


# Tạo thư mục lưu model nếu chưa có
save_dir = "/kaggle/working/NLP_saved_models"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

model_path = os.path.join(save_dir, "joint_bert_nyt.pt")

# Lưu toàn bộ trọng số của mô hình
torch.save(model.state_dict(), model_path)

print(f"Mô hình đã được lưu thành công tại: {model_path}")

> ***Hiển thị kết quả***

In [ ]:
import matplotlib.pyplot as plt

# Vẽ biểu đồ Loss
plt.plot(range(1, num_epochs + 1), history["loss"], marker='o', color='b')
plt.title('Biểu đồ độ lỗi (Training Loss) qua các Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

In [ ]:
# Cấu hình thiết bị phần cứng và Tokenizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

print("⏳ 2. Đang nạp trọng số mô hình đã lưu từ ổ đĩa...")
# Khởi tạo model với chính xác 7 nhãn thực thể và 25 nhãn quan hệ như lúc Train
loaded_model = JointBERTForRE('bert-base-cased', num_entity_labels=7, num_relation_labels=25)
saved_model_path = "/kaggle/input/models/phuocsangtran/nlp-joint/transformers/default/1/joint_bert_nyt.pt"

loaded_model.load_state_dict(torch.load(saved_model_path, map_location=device))
loaded_model.to(device)
loaded_model.eval() 


print("⏳ 3. Đang đóng gói dữ liệu tập Test vào Loader...")
test_path = '/kaggle/input/datasets/daishinkan002/new-york-times-relation-extraction-dataset/dataset/test.json'
test_dataset = NYTDataset(test_path, tokenizer, max_len=128, rel2id=rel2id, ent2id=ent2id)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

from sklearn.metrics import precision_score, recall_score, f1_score

print("⏳ 4. Mô hình đang tính toán dự đoán trên toàn bộ tập Test...")
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['relation_labels'].to(device)
        
        _, rel_logits = loaded_model(input_ids, attention_mask)
        preds = torch.argmax(rel_logits, dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

# -----------------------------------------------------------------
# A. TÍNH MACRO METRICS (Tính trung bình chia đều cho mọi nhãn)
# -----------------------------------------------------------------
macro_prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
macro_rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

# -----------------------------------------------------------------
# B. TÍNH MICRO METRICS BỎ QUA 'NA' (Chuẩn so sánh của Zheng 2017)
# -----------------------------------------------------------------
# Tạo danh sách các ID của nhãn có ý nghĩa (Loại bỏ ID của nhãn 'NA')
positive_labels = [id for name, id in rel2id.items() if name != 'NA']

micro_prec = precision_score(all_labels, all_preds, labels=positive_labels, average='micro', zero_division=0)
micro_rec = recall_score(all_labels, all_preds, labels=positive_labels, average='micro', zero_division=0)
micro_f1 = f1_score(all_labels, all_preds, labels=positive_labels, average='micro', zero_division=0)

# -----------------------------------------------------------------
# C. IN BÁO CÁO TỔNG KẾT
# -----------------------------------------------------------------
print("\n" + "="*65)
print("[MACRO METRICS] (Đánh giá độ công bằng trên mọi nhãn):")
print(f"   - Macro Precision : {macro_prec:.4f}")
print(f"   - Macro Recall    : {macro_rec:.4f}")
print(f"   - Macro F1-Score  : {macro_f1:.4f}")
print("-" * 65)
print("[MICRO METRICS BỎ QUA 'NA'] (So sánh ngang hàng Zheng 2017):")
print(f"   - Micro Precision : {micro_prec:.4f}")
print(f"   - Micro Recall    : {micro_rec:.4f}")
print(f"   - Micro F1-Score  : {micro_f1:.4f}")
print("="*65)

In [ ]:
# 1. DỮ LIỆU CỦA ZHENG ET AL. (2017) - Lấy từ cột (E1, E2)
zheng_prec = 0.645
zheng_rec = 0.437
zheng_f1 = 0.520
zheng_metrics = [zheng_prec, zheng_rec, zheng_f1]

# 2. DỮ LIỆU CỦA BẠN - Lấy từ kết quả "MICRO METRICS BỎ QUA 'NA'" bạn vừa chạy
# LƯU Ý: Bạn hãy thay 3 con số giả định dưới đây bằng kết quả thật của bạn nhé!
your_prec = micro_prec # Thay bằng số Micro Precision của bạn
your_rec = micro_rec   # Thay bằng số Micro Recall của bạn
your_f1 = micro_f1    # Thay bằng số Micro F1-Score của bạn
your_metrics = [your_prec, your_rec, your_f1]

# 3. THIẾT LẬP THÔNG SỐ BIỂU ĐỒ
labels = ['Precision', 'Recall', 'F1-Score']
x = np.arange(len(labels))  # Vị trí các nhãn trên trục x
width = 0.35                # Độ rộng của cột

fig, ax = plt.subplots(figsize=(8, 5))

# Vẽ cột cho Zheng
rects1 = ax.bar(x - width/2, zheng_metrics, width, 
                label='NovelTagging LSTM (Zheng et al.) [6]', color='#e67e22')
# Vẽ cột cho Mô hình của bạn
rects2 = ax.bar(x + width/2, your_metrics, width, 
                label='Joint BERT (Mô hình đề xuất)', color='#2ecc71')

# 4. THÊM SỐ LIỆU LÊN ĐỈNH CỘT
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontweight='bold')

autolabel(rects1)
autolabel(rects2)

# 5. TRANG TRÍ BIỂU ĐỒ
ax.set_ylabel('Giá trị độ đo', fontsize=11, fontweight='bold')
ax.set_title('So sánh hiệu năng Joint Extraction trên tập dữ liệu NYT', fontsize=13, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.0) # Đặt giới hạn trục Y từ 0 đến 1
ax.legend(loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:


# =================================================================
# BƯỚC 1: ĐỊNH NGHĨA HÀM DỰ ĐOÁN (ĐÃ TÍCH HỢP TẦNG HẬU XỬ LÝ LOGIC)
# =================================================================
id2ent = {v: k for k, v in ent2id.items()}
id2rel = {v: k for k, v in rel2id.items()}

def predict_joint_with_logic(sentence, my_model, my_tokenizer, my_device):
    my_model.eval()
    encoding = my_tokenizer(
        sentence, max_length=128, padding='max_length',
        truncation=True, return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(my_device)
    attention_mask = encoding['attention_mask'].to(my_device)
    
    with torch.no_grad():
        ent_logits, rel_logits = my_model(input_ids, attention_mask)
        
        # 1. Giải mã Quan hệ gốc từ mô hình RE
        rel_id = torch.argmax(rel_logits, dim=1).item()
        raw_predicted_rel = id2rel.get(rel_id, "NA")
        
        # 2. Giải mã Thực thể từ mô hình NER
        ent_ids = torch.argmax(ent_logits, dim=2)[0].cpu().numpy()
        tokens = my_tokenizer.convert_ids_to_tokens(input_ids[0])
        
        extracted_entities = []
        for token, ent_id in zip(tokens, ent_ids):
            if token not in ['[CLS]', '[SEP]', '[PAD]']:
                tag = id2ent.get(ent_id, "O")
                if tag != "O": 
                    if token.startswith('##'):
                        extracted_entities.append(f"{token.replace('##', '')} ({tag})")
                    else:
                        extracted_entities.append(f"{token} ({tag})")
                        
    # Nếu số lượng thực thể bóc tách < 2, bắt buộc quan hệ phải là NA
    if len(extracted_entities) < 2:
        final_relation = "NA"
        if raw_predicted_rel != "NA":
            status_note = f" ĐÃ VÁ LỖI: Ép về NA (Mô hình gốc đoán mò là: {raw_predicted_rel})"
        else:
            status_note = " Hợp lệ (Cả NER và RE đều nhận diện không có quan hệ)"
    else:
        final_relation = raw_predicted_rel
        status_note = " Hợp lệ (Đủ điều kiện cấu thành bộ ba thực thể)"
                        
    return extracted_entities, final_relation, status_note


# =================================================================
# BƯỚC 2: ĐỌC FILE TEST VÀ BỐC NGẪU NHIÊN 5 CÂU MỖI LẦN CHẠY
# =================================================================
test_path = '/kaggle/input/datasets/daishinkan002/new-york-times-relation-extraction-dataset/dataset/test.json'

test_data = []
with open(test_path, 'r', encoding='utf-8') as f:
    for line in f:
        test_data.append(json.loads(line))

# Kích hoạt tính năng ngẫu nhiên theo thời gian thực
random.seed(time.time()) 

# Bốc ngẫu nhiên 5 câu từ tập dữ liệu test
sample_items = random.sample(test_data, 5)

print("\n" + "="*85)
print(" ĐÁNH GIÁ 5 CÂU NGẪU NHIÊN - ĐÃ CÓ TẦNG BẢO VỆ LOGIC (POST-PROCESSING)")
print("="*85)

for i, item in enumerate(sample_items):
    sentence = item['sentText']
    
    # --- LẤY NHÃN THỰC TẾ (GROUND TRUTH) ---
    true_rels = item.get('relationMentions', [])
    true_rel_str = true_rels[0]['label'] if true_rels else "NA"
    true_ents = [f"{ent['text']} ({ent['label']})" for ent in item.get('entityMentions', [])]
    
    # --- LẤY DỰ ĐOÁN (PREDICTION) TỪ HÀM ĐÃ ĐƯỢC FIX ---
    pred_entities, pred_relation, status_note = predict_joint_with_logic(sentence, loaded_model, tokenizer, device)
    
    # --- IN KẾT QUẢ SO SÁNH TRỰC QUAN ---
    print(f"\n CÂU NGẪU NHIÊN SỐ {i+1}")
    print(f" Câu văn: \"{sentence}\"")
    print("-" * 70)
    print(f" THỰC TẾ GỐC :  Thực thể: {true_ents}")
    print(f"                  Quan hệ:  {true_rel_str}")
    print(f" AI DỰ ĐOÁN  :  Thực thể: {pred_entities}")
    print(f"                  Quan hệ:  {pred_relation}")
    print(f" NHẬT KÝ LỌC : {status_note}")
    
    # Đánh giá nhanh bằng mắt sau khi đã áp dụng logic
    status = "TRÙNG KHỚP HOÀN HẢO!" if true_rel_str == pred_relation else "CÓ SỰ KHÁC BIỆT / SAI LỆCH!"
    print(f" ĐÁNH GIÁ CHUNG: {status}")
    print("="*85)